# 04 — Sensitivity Analysis and Statistical Inference

This notebook presents Sobol global sensitivity results, bootstrap/Bayesian inference,
and Decision Curve Analysis in detail.

In [ ]:
import pandas as pd
import json
from pathlib import Path
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

## Sobol Global Sensitivity Analysis

Total-order indices (ST) quantify each parameter's contribution to outcome variance,
including interaction effects.

In [ ]:
outcomes = ['unsafe_detection_rate', 'safe_throughput', 'false_negative_harm', 'mean_total_friction']
for outcome in outcomes:
    df = pd.read_csv(f'outputs/tables/sobol_{outcome}.csv')
    print(f'\n{outcome.replace("_", " ").title()}')
    print(df[['parameter', 'ST', 'status', 'rank']].to_string(index=False))

## Statistical Inference

Bootstrap confidence intervals (500 resamples) and Bayesian Beta-posterior credible intervals
for the moderate threshold profile across replicates.

In [ ]:
inf = json.loads(Path('outputs/processed/inference_results.json').read_text())
print(f"Detection: {inf['detection_mean']:.4f} [{inf['detection_ci_lower']:.4f}, {inf['detection_ci_upper']:.4f}]")
print(f"Throughput: {inf['throughput_mean']:.4f} [{inf['throughput_ci_lower']:.4f}, {inf['throughput_ci_upper']:.4f}]")
print(f"FN Harm: {inf['harm_mean']:.2f} [{inf['harm_ci_lower']:.2f}, {inf['harm_ci_upper']:.2f}]")
print(f"Friction: {inf['friction_mean']:.4f} [{inf['friction_ci_lower']:.4f}, {inf['friction_ci_upper']:.4f}]")
print(f"\nBayesian detection: {inf['detection_bayesian']['posterior_mean']:.4f} [{inf['detection_bayesian']['ci_lower']:.4f}, {inf['detection_bayesian']['ci_upper']:.4f}]")
print(f"Bayesian throughput: {inf['throughput_bayesian']['posterior_mean']:.4f} [{inf['throughput_bayesian']['ci_lower']:.4f}, {inf['throughput_bayesian']['ci_upper']:.4f}]")

## Scenario Variation

Profile performance across 5 scenario families.

In [ ]:
scenarios = pd.read_csv('outputs/raw/scenario_results.csv')
for scenario_idx in sorted(scenarios['scenario_idx'].unique()):
    s = scenarios[scenarios['scenario_idx'] == scenario_idx].iloc[0]
    print(f"\nScenario {scenario_idx + 1}: UBR={s['unsafe_base_rate']}, noise={s['auditability_noise']}, evidence={s['evidence_regime']}, drift={s['drift_regime']}")
    for profile in ['permissive', 'moderate', 'strict', 'very_strict']:
        p = scenarios[(scenarios['scenario_idx'] == scenario_idx) & (scenarios['profile_name'] == profile)]
        if len(p) > 0:
            print(f"  {profile:12s}: det={p['unsafe_detection_rate'].mean():.3f}, thr={p['safe_throughput'].mean():.3f}, harm={p['false_negative_harm'].mean():.1f}")

## Sensitivity Assertions

In [ ]:
# Verify Sobol key claims
sobol_det = pd.read_csv('outputs/tables/sobol_unsafe_detection_rate.csv')
sobol_harm = pd.read_csv('outputs/tables/sobol_false_negative_harm.csv')
sobol_thr = pd.read_csv('outputs/tables/sobol_safe_throughput.csv')
sobol_fric = pd.read_csv('outputs/tables/sobol_mean_total_friction.csv')

assert float(sobol_det[sobol_det['parameter']=='safety_gate']['ST']) == 1.0, 'Safety ST for detection != 1.0'
assert float(sobol_harm[sobol_harm['parameter']=='safety_gate']['ST']) == 1.0, 'Safety ST for harm != 1.0'
assert abs(float(sobol_thr[sobol_thr['parameter']=='calibration_gate']['ST']) - 0.32) < 0.02, 'Calibration ST for throughput != ~0.32'
assert abs(float(sobol_fric[sobol_fric['parameter']=='safety_gate']['ST']) - 0.23) < 0.02, 'Safety ST for friction != ~0.23'

# Verify inference claims
assert abs(inf['detection_mean'] - 0.996) < 0.002, f"Detection mean {inf['detection_mean']} not ~0.996"
assert abs(inf['throughput_mean'] - 0.210) < 0.002, f"Throughput mean {inf['throughput_mean']} not ~0.210"

print('All sensitivity and inference assertions PASSED')